# Importation des données

In [1]:
import os
import subprocess
import pandas as pd

## Chemin d'accès aux données controlées du `CDR`

Lien vers CDR pour l'accès aux données génétiques -> https://support.researchallofus.org/hc/en-us/articles/29475233432212-Controlled-CDR-Directory

In [2]:
# Définit le chemin vers les fichiers auxiliaires SNP/Indel dans le bucket Google Cloud
auxiliary_path = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux"

## Données `ancestry_pred`

In [3]:
# Télécharge le fichier d'ancestry depuis le CDR vers le répertoire jupyter
!gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv .

Copying gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv...
/ [1 files][163.4 MiB/163.4 MiB]                                                
Operation completed over 1 objects/163.4 MiB.                                    


In [4]:
# Charge le fichier contenant les prédictions d'origine ancestrale dans un DataFrame
ancestry_pred = pd.read_csv("ancestry_preds.tsv", delimiter="\t")
ancestry_pred

,research_id,ancestry_pred,probabilities,pca_features,ancestry_pred_other
0,1000000,afr,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[-0.29356093859992993, -0.006344531427051658, ...",afr
1,1000004,eur,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0]","[0.10130837407349723, 0.13870298220174238, 0.0...",eur
2,1000033,eur,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0]","[0.09848604976046149, 0.1245991833533566, 0.00...",eur
3,1000039,afr,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[-0.26713841551900935, 0.00469795004195689, 0....",afr
4,1000042,afr,"[0.99, 0.01, 0.0, 0.0, 0.0, 0.0]","[-0.2562773941608334, 0.004901392894403225, -0...",afr
...,...,...,...,...,...
414825,9999678,eur,"[0.0, 0.0, 0.0, 0.99, 0.0, 0.01]","[0.09874274635571259, 0.13176851496404074, 0.0...",eur
414826,9999697,amr,"[0.0, 1.0, 0.0, 0.0, 0.0, 0.0]","[0.08506698802084682, 0.028602321968432515, -0...",amr
414827,9999715,eur,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0]","[0.0995298469352078, 0.13283576852436813, 0.00...",eur
414828,9999755,eur,"[0.02, 0.17, 0.02, 0.64, 0.12, 0.03]","[0.06449853461869955, 0.11663708597332126, 0.0...",oth


### Formatage des données `pca_features`

In [5]:
# Supprime les crochets '[' et ']' autour de la chaîne de caractères dans la colonne 'pca_features'
ancestry_pred["pca_features"] = ancestry_pred["pca_features"].str[1:-1]

In [6]:
# Sépare les 16 composantes principales (PCA) encodées en chaîne de caractères et les convertit en float
PCs = ancestry_pred["pca_features"].str.split(",", n=16, expand=True)
PCs = PCs.astype(float)

In [7]:
# Extrait la colonne contenant les identifiants de recherche des individus
pid = ancestry_pred[["research_id"]]

In [8]:
# Renomme les colonnes du DataFrame 'PCs' avec les noms des 16 premières composantes principales (PC1 à PC16)
columns_pca = ["PC1", "PC2", "PC3", "PC4", "PC5", "PC6", "PC7", "PC8", "PC9", "PC10", "PC11", "PC12", "PC13", "PC14", "PC15", "PC16"]
PCs.columns = columns_pca

In [9]:
# Combine les identifiants 'research_id' avec les composantes principales dans un seul DataFrame final
PCs_final = pd.concat([pid, PCs], axis=1)
PCs_final.head(5)

,research_id,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16
0,1000000,-0.293561,-0.006345,0.002386,0.001446,0.024304,-0.001529,-0.005621,-0.001204,-0.000920,0.006886,0.004709,0.004915,0.011461,0.002675,-0.001492,0.000896
1,1000004,0.101308,0.138703,0.006683,0.053012,0.003345,0.019714,-0.011616,-0.001016,-0.001095,-0.000909,-0.001277,-0.000694,-0.000668,-0.000960,-0.001257,0.000115
2,1000033,0.098486,0.124599,0.009398,0.042617,0.003846,0.026659,-0.018481,-0.001463,-0.001203,0.000465,0.000469,0.000629,0.000019,-0.000210,-0.000013,0.000411
3,1000039,-0.267138,0.004698,0.001046,0.001767,0.031375,0.001713,-0.009450,0.016385,-0.000844,0.001511,0.003544,-0.000247,-0.010346,0.005279,-0.013716,0.008376
4,1000042,-0.256277,0.004901,-0.002448,0.009514,0.008931,0.010683,0.003820,-0.002210,0.010388,-0.008314,-0.002740,0.003094,0.007370,0.001563,-0.002620,0.006765


In [10]:
# Fusionne les métadonnées d'ancestry sans la colonne PCA brute avec les composantes principales (PCs)
df_ancestry = ancestry_pred.drop(columns='pca_features').merge(PCs_final, on='research_id', how='inner')
df_ancestry

,research_id,ancestry_pred,probabilities,ancestry_pred_other,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16
0,1000000,afr,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]",afr,-0.293561,-0.006345,0.002386,0.001446,0.024304,-0.001529,-0.005621,-0.001204,-0.000920,0.006886,0.004709,0.004915,0.011461,0.002675,-0.001492,0.000896
1,1000004,eur,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0]",eur,0.101308,0.138703,0.006683,0.053012,0.003345,0.019714,-0.011616,-0.001016,-0.001095,-0.000909,-0.001277,-0.000694,-0.000668,-0.000960,-0.001257,0.000115
2,1000033,eur,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0]",eur,0.098486,0.124599,0.009398,0.042617,0.003846,0.026659,-0.018481,-0.001463,-0.001203,0.000465,0.000469,0.000629,0.000019,-0.000210,-0.000013,0.000411
3,1000039,afr,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]",afr,-0.267138,0.004698,0.001046,0.001767,0.031375,0.001713,-0.009450,0.016385,-0.000844,0.001511,0.003544,-0.000247,-0.010346,0.005279,-0.013716,0.008376
4,1000042,afr,"[0.99, 0.01, 0.0, 0.0, 0.0, 0.0]",afr,-0.256277,0.004901,-0.002448,0.009514,0.008931,0.010683,0.003820,-0.002210,0.010388,-0.008314,-0.002740,0.003094,0.007370,0.001563,-0.002620,0.006765
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
414825,9999678,eur,"[0.0, 0.0, 0.0, 0.99, 0.0, 0.01]",eur,0.098743,0.131769,0.010411,0.048562,0.002988,0.021923,-0.017846,-0.002502,-0.000634,0.000194,0.000252,-0.001290,0.001715,-0.000752,-0.000415,-0.000020
414826,9999697,amr,"[0.0, 1.0, 0.0, 0.0, 0.0, 0.0]",amr,0.085067,0.028602,-0.107409,0.005666,0.000069,-0.012026,0.010985,0.002458,0.001513,-0.001535,-0.001271,-0.001320,-0.000950,-0.001408,-0.003327,0.000309
414827,9999715,eur,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0]",eur,0.099530,0.132836,0.008332,0.050046,0.003166,0.023643,-0.015716,-0.002708,-0.001984,-0.001566,0.000294,0.000889,-0.001221,0.001146,-0.000401,0.000165
414828,9999755,eur,"[0.02, 0.17, 0.02, 0.64, 0.12, 0.03]",oth,0.064499,0.116637,0.009492,0.042075,-0.005982,-0.022958,0.020151,0.001276,0.000953,-0.000316,-0.003218,0.000905,-0.000271,0.000847,0.001971,-0.000443


### Formatage des données `probabilities`

In [11]:
# Supprime les crochets '[' et ']' autour de la chaîne de caractères dans la colonne 'probabilities'
df_ancestry["probabilities"] = df_ancestry["probabilities"].str[1:-1]

In [12]:
# Sépare les 6 ethnies encodées en chaîne de caractères et les convertit en float
probabilities = df_ancestry["probabilities"].str.split(",", n=6, expand=True)
probabilities = probabilities.astype(float)

In [13]:
# Renomme les colonnes du DataFrame 'probabilities' avec les noms des 6 ethnies
columns_probabilities = ["afr_ancestry_prob", "amr_ancestry_prob", "eas_ancestry_prob", "eur_ancestry_prob", "mid_ancestry_prob", "sas_ancestry_prob"]
probabilities.columns = columns_probabilities

In [14]:
# Combine les identifiants 'research_id' avec les ethnies dans un seul DataFrame final
probabilities_final = pd.concat([pid, probabilities], axis=1)
probabilities_final.head(5)

,research_id,afr_ancestry_prob,amr_ancestry_prob,eas_ancestry_prob,eur_ancestry_prob,mid_ancestry_prob,sas_ancestry_prob
0,1000000,1.00,0.00,0.0,0.0,0.0,0.0
1,1000004,0.00,0.00,0.0,1.0,0.0,0.0
2,1000033,0.00,0.00,0.0,1.0,0.0,0.0
3,1000039,1.00,0.00,0.0,0.0,0.0,0.0
4,1000042,0.99,0.01,0.0,0.0,0.0,0.0


In [15]:
# Fusionne les métadonnées d'ancestry sans la colonne PCA brute avec les composantes principales (PCs)
df_ancestry_final = df_ancestry.drop(columns='probabilities').merge(probabilities_final, on='research_id', how='inner')
df_ancestry_final = df_ancestry_final[['research_id','ancestry_pred'] + columns_probabilities + ['ancestry_pred_other'] + columns_pca]
df_ancestry_final

,research_id,ancestry_pred,afr_ancestry_prob,amr_ancestry_prob,eas_ancestry_prob,eur_ancestry_prob,mid_ancestry_prob,sas_ancestry_prob,ancestry_pred_other,PC1,...,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16
0,1000000,afr,1.00,0.00,0.00,0.00,0.00,0.00,afr,-0.293561,...,-0.005621,-0.001204,-0.000920,0.006886,0.004709,0.004915,0.011461,0.002675,-0.001492,0.000896
1,1000004,eur,0.00,0.00,0.00,1.00,0.00,0.00,eur,0.101308,...,-0.011616,-0.001016,-0.001095,-0.000909,-0.001277,-0.000694,-0.000668,-0.000960,-0.001257,0.000115
2,1000033,eur,0.00,0.00,0.00,1.00,0.00,0.00,eur,0.098486,...,-0.018481,-0.001463,-0.001203,0.000465,0.000469,0.000629,0.000019,-0.000210,-0.000013,0.000411
3,1000039,afr,1.00,0.00,0.00,0.00,0.00,0.00,afr,-0.267138,...,-0.009450,0.016385,-0.000844,0.001511,0.003544,-0.000247,-0.010346,0.005279,-0.013716,0.008376
4,1000042,afr,0.99,0.01,0.00,0.00,0.00,0.00,afr,-0.256277,...,0.003820,-0.002210,0.010388,-0.008314,-0.002740,0.003094,0.007370,0.001563,-0.002620,0.006765
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
414825,9999678,eur,0.00,0.00,0.00,0.99,0.00,0.01,eur,0.098743,...,-0.017846,-0.002502,-0.000634,0.000194,0.000252,-0.001290,0.001715,-0.000752,-0.000415,-0.000020
414826,9999697,amr,0.00,1.00,0.00,0.00,0.00,0.00,amr,0.085067,...,0.010985,0.002458,0.001513,-0.001535,-0.001271,-0.001320,-0.000950,-0.001408,-0.003327,0.000309
414827,9999715,eur,0.00,0.00,0.00,1.00,0.00,0.00,eur,0.099530,...,-0.015716,-0.002708,-0.001984,-0.001566,0.000294,0.000889,-0.001221,0.001146,-0.000401,0.000165
414828,9999755,eur,0.02,0.17,0.02,0.64,0.12,0.03,oth,0.064499,...,0.020151,0.001276,0.000953,-0.000316,-0.003218,0.000905,-0.000271,0.000847,0.001971,-0.000443


### Exportation des données

In [16]:
# Enregistre le DataFrame final dans un fichier TSV local
destination_filename = 'df_ancestry_final.tsv'
df_ancestry_final.to_csv(destination_filename, index=False)

# Récupère le nom du bucket de l'environnement d'exécution
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Copie le fichier TSV local vers le dossier "Data" du bucket Google Cloud
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

# Affiche les éventuels messages d’erreur de la commande gsutil
output.stderr

b'Copying file://./df_ancestry_final.tsv [Content-Type=text/tab-separated-values]...\n/ [0 files][    0.0 B/154.0 MiB]                                                \r==> NOTE: You are uploading one or more large file(s), which would run\nsignificantly faster if you enable parallel composite uploads. This\nfeature can be enabled by editing the\n"parallel_composite_upload_threshold" value in your .boto\nconfiguration file. However, note that if you do this large files will\nbe uploaded as `composite objects\n<https://cloud.google.com/storage/docs/composite-objects>`_,which\nmeans that any user who downloads such objects will need to have a\ncompiled crcmod installed (see "gsutil help crcmod"). This is because\nwithout a compiled crcmod, computing checksums on composite objects is\nso slow that gsutil disables downloads of composite objects.\n\n-\r- [0 files][  2.1 MiB/154.0 MiB]                                                \r\\\r\\ [0 files][  4.4 MiB/154.0 MiB]                      

## Données `aou_admixture_estimates_rye_v8` 

In [17]:
# Télécharge le fichier d'estimations d'admixture depuis le CDR
admixture_estimates_q = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/admixture_estimates/aou_admixture_estimates_rye_v8.Q"
!gsutil -m -u $$GOOGLE_PROJECT cp $admixture_estimates_q .

Copying gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/admixture_estimates/aou_admixture_estimates_rye_v8.Q...
- [1/1 files][ 32.4 MiB/ 32.4 MiB] 100% Done                                    
Operation completed over 1 objects/32.4 MiB.                                     


In [18]:
# Charge les estimations d'admixture dans un DataFrame et réinitialise l'index
df_rye = pd.read_csv('aou_admixture_estimates_rye_v8.Q', sep='\t', index_col='research_id').reset_index()
df_rye

,research_id,eur,eas,amr,afr,sas,mid
0,1000000,0.000000,0.002611,0.000000,0.940905,0.000000,0.056484
1,1000004,0.892052,0.000000,0.063450,0.000000,0.000000,0.044498
2,1000033,0.886335,0.000000,0.039803,0.000000,0.073862,0.000000
3,1000039,0.045004,0.003594,0.000000,0.874350,0.003528,0.073524
4,1000042,0.115799,0.014158,0.000000,0.853614,0.000000,0.016430
...,...,...,...,...,...,...,...
414825,9999678,0.913647,0.000000,0.028156,0.000000,0.029290,0.028907
414826,9999697,0.000000,0.000000,0.947722,0.000000,0.032638,0.019640
414827,9999715,0.913319,0.000000,0.044853,0.000000,0.018338,0.023490
414828,9999755,0.604467,0.000000,0.020042,0.052073,0.000000,0.323418


In [19]:
# Renomme les colonnes d'origine pour indiquer qu'elles proviennent du fichier RYE
df_rye.rename(columns={'eur':'eur_rye',
                       'eas':'eas_rye',
                       'amr':'amr_rye',
                       'afr':'afr_rye',
                       'sas':'sas_rye',
                       'mid':'mid_rye'}, inplace=True)
df_rye

,research_id,eur_rye,eas_rye,amr_rye,afr_rye,sas_rye,mid_rye
0,1000000,0.000000,0.002611,0.000000,0.940905,0.000000,0.056484
1,1000004,0.892052,0.000000,0.063450,0.000000,0.000000,0.044498
2,1000033,0.886335,0.000000,0.039803,0.000000,0.073862,0.000000
3,1000039,0.045004,0.003594,0.000000,0.874350,0.003528,0.073524
4,1000042,0.115799,0.014158,0.000000,0.853614,0.000000,0.016430
...,...,...,...,...,...,...,...
414825,9999678,0.913647,0.000000,0.028156,0.000000,0.029290,0.028907
414826,9999697,0.000000,0.000000,0.947722,0.000000,0.032638,0.019640
414827,9999715,0.913319,0.000000,0.044853,0.000000,0.018338,0.023490
414828,9999755,0.604467,0.000000,0.020042,0.052073,0.000000,0.323418


In [20]:
# Enregistre le DataFrame `df_rye` dans un fichier TSV local
destination_filename = 'df_rye_final.tsv'
df_rye.to_csv(destination_filename, index=False)

# Récupère le nom du bucket Google Cloud depuis la variable d’environnement
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Copie le fichier TSV local dans le dossier "Data" du bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

# Affiche les éventuelles erreurs retournées par gsutil
output.stderr

b'Copying file://./df_rye_final.tsv [Content-Type=text/tab-separated-values]...\n/ [0 files][    0.0 B/ 32.4 MiB]                                                \r-\r- [0 files][  3.1 MiB/ 32.4 MiB]                                                \r\\\r\\ [0 files][  4.9 MiB/ 32.4 MiB]                                                \r|\r/\r/ [0 files][  6.7 MiB/ 32.4 MiB]                                                \r-\r- [0 files][  8.5 MiB/ 32.4 MiB]                                                \r\\\r|\r| [0 files][ 10.3 MiB/ 32.4 MiB]                                                \r/\r/ [0 files][ 12.1 MiB/ 32.4 MiB]                                                \r-\r- [0 files][ 13.9 MiB/ 32.4 MiB]                                                \r\\\r|\r| [0 files][ 15.7 MiB/ 32.4 MiB]                                                \r/\r/ [0 files][ 17.5 MiB/ 32.4 MiB]                                                \r-\r\\\r\\ [0 files][ 19.3 MiB/ 32.4 MiB]                   

## Données `relatedness_flagged_samples`

In [21]:
# Télécharge le fichier contenant les échantillons apparentés depuis le bucket Google Cloud
related_samples_path = 'gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/relatedness/relatedness_flagged_samples.tsv'
!gsutil -u $$GOOGLE_PROJECT cp $related_samples_path .

Copying gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/relatedness/relatedness_flagged_samples.tsv...
/ [1 files][239.0 KiB/239.0 KiB]                                                
Operation completed over 1 objects/239.0 KiB.                                    


In [22]:
# Charge le fichier des échantillons apparentés dans un DataFrame
df_relatedness = pd.read_csv('relatedness_flagged_samples.tsv', sep='\t')
df_relatedness

,sample_id
0,1000039
1,1000091
2,1000109
3,1000127
4,1000320
...,...
30579,9996831
30580,9996887
30581,9997301
30582,9998505


In [23]:
# Enregistre le DataFrame `df_relatedness` dans un fichier TSV local
destination_filename = 'df_relatedness_final.tsv'
df_relatedness.to_csv(destination_filename, index=False)

# Récupère le nom du bucket Google Cloud depuis la variable d’environnement
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Copie le fichier TSV local dans le dossier "Data" du bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

# Affiche les éventuelles erreurs retournées par gsutil
output.stderr

b'Copying file://./df_relatedness_final.tsv [Content-Type=text/tab-separated-values]...\n/ [0 files][    0.0 B/239.0 KiB]                                                \r/ [1 files][239.0 KiB/239.0 KiB]                                                \r\nOperation completed over 1 objects/239.0 KiB.                                    \n'

## Données `plink_bed_files_array`

In [24]:
# Téléchargement du fichier .fam (informations d'identité des individus) depuis le bucket Google Cloud
!gsutil -m -u $$GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam .

Copying gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam...
/ [1/1 files][  8.1 MiB/  8.1 MiB] 100% Done                                    
Operation completed over 1 objects/8.1 MiB.                                      


In [25]:
df_plink_fam_files_array = pd.read_csv("arrays.fam", delimiter = "\t", names = ["A","B","C","D","E","F"])
df_plink_fam_files_array

,A,B,C,D,E,F
0,0,1000000,0,0,0,NaN
1,0,1000004,0,0,0,NaN
2,0,1000033,0,0,0,NaN
3,0,1000039,0,0,0,NaN
4,0,1000042,0,0,0,NaN
...,...,...,...,...,...,...
447273,0,9999678,0,0,0,NaN
447274,0,9999697,0,0,0,NaN
447275,0,9999715,0,0,0,NaN
447276,0,9999755,0,0,0,NaN


In [26]:
# Enregistre le DataFrame `df_relatedness` dans un fichier TSV local
destination_filename = 'df_plink_fam_files_array.tsv'
df_plink_fam_files_array.to_csv(destination_filename, index=False)

# Récupère le nom du bucket Google Cloud depuis la variable d’environnement
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Copie le fichier TSV local dans le dossier "Data" du bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

# Affiche les éventuelles erreurs retournées par gsutil
output.stderr

b'Copying file://./df_plink_fam_files_array.tsv [Content-Type=text/tab-separated-values]...\n/ [0 files][    0.0 B/  7.2 MiB]                                                \r-\r- [0 files][  2.1 MiB/  7.2 MiB]                                                \r\\\r|\r| [0 files][  4.4 MiB/  7.2 MiB]                                                \r/\r/ [0 files][  6.4 MiB/  7.2 MiB]                                                \r-\r- [1 files][  7.2 MiB/  7.2 MiB]                                                \r\\\r\nOperation completed over 1 objects/7.2 MiB.                                      \n'

## Cohorte Cancer - Femme (Controlled Tier Dataset v8)

### With Breast Cancer (BC)

In [27]:
# This query represents dataset "Woman BC invasive" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_13503019_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (44819434, 1567537)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 0 
                        AND is_selectable = 1) 
                    AND is_standard = 0 )) criteria ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.person` p 
            WHERE
                gender_concept_id IN (45878463) ) )"""

# Exécute la requête et charge les données dans un DataFrame via BigQuery
df_with_breast_cancer = pd.read_gbq(
    dataset_13503019_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

df_with_breast_cancer.head(5)

Downloading:   0%|          | 0/11739 [00:00<?, ?rows/s]

,person_id,gender_concept_id,gender,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
0,2011636,45878463,Female,1937-06-15 00:00:00+00:00,8657,American Indian or Alaska Native,38003564,Not Hispanic or Latino,45878463,Female,8657,American Indian or Alaska Native
1,7280114,45878463,Female,1948-06-15 00:00:00+00:00,8657,American Indian or Alaska Native,38003564,Not Hispanic or Latino,45878463,Female,8657,American Indian or Alaska Native
2,2253660,45878463,Female,1956-06-15 00:00:00+00:00,8657,American Indian or Alaska Native,38003564,Not Hispanic or Latino,45878463,Female,8657,American Indian or Alaska Native
3,5832436,45878463,Female,1952-06-15 00:00:00+00:00,8657,American Indian or Alaska Native,38003564,Not Hispanic or Latino,45878463,Female,8657,American Indian or Alaska Native
4,1340234,45878463,Female,1960-06-15 00:00:00+00:00,8657,American Indian or Alaska Native,38003564,Not Hispanic or Latino,45878463,Female,8657,American Indian or Alaska Native


In [28]:
# Sélection des colonnes d'intérêt parmi les données issues de la requête sur le cancer du sein invasif
columns_bc_data = ['person_id', 'gender', 'date_of_birth', 'race', 'ethnicity', 'self_reported_category']

# Filtrage du DataFrame pour ne conserver que les colonnes `columns_bc_data`
df_with_breast_cancer = df_with_breast_cancer[columns_bc_data]
df_with_breast_cancer

,person_id,gender,date_of_birth,race,ethnicity,self_reported_category
0,2011636,Female,1937-06-15 00:00:00+00:00,American Indian or Alaska Native,Not Hispanic or Latino,American Indian or Alaska Native
1,7280114,Female,1948-06-15 00:00:00+00:00,American Indian or Alaska Native,Not Hispanic or Latino,American Indian or Alaska Native
2,2253660,Female,1956-06-15 00:00:00+00:00,American Indian or Alaska Native,Not Hispanic or Latino,American Indian or Alaska Native
3,5832436,Female,1952-06-15 00:00:00+00:00,American Indian or Alaska Native,Not Hispanic or Latino,American Indian or Alaska Native
4,1340234,Female,1960-06-15 00:00:00+00:00,American Indian or Alaska Native,Not Hispanic or Latino,American Indian or Alaska Native
...,...,...,...,...,...,...
11734,1951816,Female,1978-06-15 00:00:00+00:00,White,Not Hispanic or Latino,White
11735,1390558,Female,1979-06-15 00:00:00+00:00,White,Not Hispanic or Latino,White
11736,8233973,Female,1981-06-15 00:00:00+00:00,White,Not Hispanic or Latino,White
11737,2877487,Female,1988-06-15 00:00:00+00:00,White,Not Hispanic or Latino,White


In [29]:
# Ajout d'une colonne binaire indiquant la présence d'un cancer du sein (1 = atteint)
df_with_breast_cancer['has_BC'] = 1

In [30]:
# Enregistre le DataFrame `df_with_breast_cancer` dans un fichier TSV local
destination_filename = 'df_with_breast_cancer_final.tsv'
df_with_breast_cancer.to_csv(destination_filename, index=False)

# Récupère le nom du bucket Google Cloud depuis la variable d’environnement
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Copie le fichier TSV local dans le dossier "Data" du bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

# Affiche les éventuelles erreurs retournées par gsutil
output.stderr

b'Copying file://./df_with_breast_cancer_final.tsv [Content-Type=text/tab-separated-values]...\n/ [0 files][    0.0 B/ 1012 KiB]                                                \r/ [1 files][ 1012 KiB/ 1012 KiB]                                                \r-\r\nOperation completed over 1 objects/1012.5 KiB.                                   \n'

### NOT Breast Cancer (BC)

In [31]:
# This query represents dataset "Women NON BC invasive" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_08407181_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.person` p 
            WHERE
                gender_concept_id IN (45878463) ) 
            AND cb_search_person.person_id NOT IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (44819434, 1567537)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 0 
                        AND is_selectable = 1) 
                    AND is_standard = 0 )) criteria ) )"""

# Exécute la requête et charge les données dans un DataFrame via BigQuery
df_not_breast_cancer = pd.read_gbq(
    dataset_08407181_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

df_not_breast_cancer.head(5)

Downloading:   0%|          | 0/379071 [00:00<?, ?rows/s]

,person_id,gender_concept_id,gender,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
0,1104573,45878463,Female,1996-06-15 00:00:00+00:00,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,1177221,I prefer not to answer,45882607,None of these
1,9817954,45878463,Female,1979-06-15 00:00:00+00:00,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,1177221,I prefer not to answer,45882607,None of these
2,1163679,45878463,Female,1965-06-15 00:00:00+00:00,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,1177221,I prefer not to answer,45882607,None of these
3,9158204,45878463,Female,1988-06-15 00:00:00+00:00,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,1177221,I prefer not to answer,45882607,None of these
4,2300102,45878463,Female,1951-06-15 00:00:00+00:00,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,0,No matching concept,45882607,None of these


In [32]:
# Filtrage du DataFrame pour ne conserver que les colonnes `columns_bc_data`
df_not_breast_cancer = df_not_breast_cancer[columns_bc_data]
df_not_breast_cancer

,person_id,gender,date_of_birth,race,ethnicity,self_reported_category
0,1104573,Female,1996-06-15 00:00:00+00:00,None of these,What Race Ethnicity: Race Ethnicity None Of These,None of these
1,9817954,Female,1979-06-15 00:00:00+00:00,None of these,What Race Ethnicity: Race Ethnicity None Of These,None of these
2,1163679,Female,1965-06-15 00:00:00+00:00,None of these,What Race Ethnicity: Race Ethnicity None Of These,None of these
3,9158204,Female,1988-06-15 00:00:00+00:00,None of these,What Race Ethnicity: Race Ethnicity None Of These,None of these
4,2300102,Female,1951-06-15 00:00:00+00:00,None of these,What Race Ethnicity: Race Ethnicity None Of These,None of these
...,...,...,...,...,...,...
379066,5683164,Female,1966-06-15 00:00:00+00:00,None Indicated,Hispanic or Latino,What Race Ethnicity: Hispanic
379067,7834223,Female,1981-06-15 00:00:00+00:00,None Indicated,Hispanic or Latino,What Race Ethnicity: Hispanic
379068,2022233,Female,1946-06-15 00:00:00+00:00,None Indicated,Hispanic or Latino,What Race Ethnicity: Hispanic
379069,1549231,Female,1983-06-15 00:00:00+00:00,None Indicated,Hispanic or Latino,What Race Ethnicity: Hispanic


In [33]:
# Ajout d'une colonne binaire indiquant l'absence de cancer du sein (0 = non atteint)
df_not_breast_cancer['has_BC'] = 0

In [34]:
# Enregistre le DataFrame `df_with_breast_cancer` dans un fichier TSV local
destination_filename = 'df_not_breast_cancer_final.tsv'
df_not_breast_cancer.to_csv(destination_filename, index=False)

# Récupère le nom du bucket Google Cloud depuis la variable d’environnement
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Copie le fichier TSV local dans le dossier "Data" du bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

# Affiche les éventuelles erreurs retournées par gsutil
output.stderr

b'Copying file://./df_not_breast_cancer_final.tsv [Content-Type=text/tab-separated-values]...\n/ [0 files][    0.0 B/ 33.4 MiB]                                                \r-\r- [0 files][  3.1 MiB/ 33.4 MiB]                                                \r\\\r\\ [0 files][  4.9 MiB/ 33.4 MiB]                                                \r|\r/\r/ [0 files][  6.7 MiB/ 33.4 MiB]                                                \r-\r- [0 files][  8.5 MiB/ 33.4 MiB]                                                \r\\\r|\r| [0 files][ 10.3 MiB/ 33.4 MiB]                                                \r/\r/ [0 files][ 12.1 MiB/ 33.4 MiB]                                                \r-\r- [0 files][ 13.9 MiB/ 33.4 MiB]                                                \r\\\r|\r| [0 files][ 15.7 MiB/ 33.4 MiB]                                                \r/\r/ [0 files][ 17.5 MiB/ 33.4 MiB]                                                \r-\r\\\r\\ [0 files][ 19.3 MiB/ 33.4 MiB]     